In [ ]:
#!pip install langchain-openai langchain_community faiss-cpu langchain_groq langchain-google-genai

#Agents

In [ ]:
import os

In [ ]:
os.environ["OPENAI_API_KEY"] =  "SUA CHAVE DE API"

In [ ]:
from langchain_openai import ChatOpenAI

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2

    )

In [ ]:
llm.invoke("hello")

In [ ]:
for i in llm.stream("hello"):
  print(i)

In [ ]:
from langchain_openai import OpenAIEmbeddings

model_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large",
)

In [ ]:
result = model_embeddings.embed_query("hello, how Can I you?")

In [ ]:
len(result)

3072

In [ ]:
import json
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# 1. ler o arquivo JSON
with open("/home/pessoas.json", "r", encoding="utf-8") as f:
    textos = json.load(f)  # isso já vira uma lista de strings

print(f"Carregado {len(textos)} textos")

Carregado 200 textos


In [ ]:
textos[:2]

['Seu nome é Beatriz Maria Silva, é uma enfermeira que trabalha em um hospital público e está especializada em saúde mental. Beatriz formou-se em enfermagem pela Universidade Federal do Rio de Janeiro e também fez um mestrado em Psiquiatria pela Universidade de São Paulo. Ela trabalhou em clínicas particulares e hospitais filantrópicos antes de se mudar para o hospital onde atualmente trabalha.',
 'Maria José Silva é uma enfermeira especialista em cuidados intensivos, formada em Enfermagem pela Faculdade de Ciências Médicas da Universidade Estadual de Campinas, onde também concluiu pós-graduação em Gestão de Serviços de Saúde. Com 10 anos de experiência em unidades de terapia intensiva, ela tem uma forte reputação por sua habilidade em lidar com casos críticos e sua capacidade de gerenciar equipes de saúde. Ela trabalhou em hospitais de grande porte em São Paulo e atualmente exerce a função de chefe de enfermagem em um dos principais centros médicos da cidade.']

In [ ]:
# 3. criar FAISS direto dos textos
vectorstore = FAISS.from_texts(
    textos,
    model_embeddings
)

In [ ]:
result = vectorstore.similarity_search("preciso de pessoas pra criar um predio", k=10)
print(result)

[Document(id='94304760-0ad5-4f77-8a03-4af1dfbfab2b', metadata={}, page_content='O nome completo da pessoa é João Pedro Silva, um arquiteto com formação em Engenharia de Construção Civil pelo Instituto Tecnológico de Aeronáutica, onde obteve sua graduação em 2010. Ele trabalhou em várias empresas de construção civil, incluindo o cargo de gerente de projetos em uma empresa líder no setor por 5 anos antes de criar sua própria empresa de arquitetura em 2015. Hoje, João Pedro é um renomado arquiteto com projetos de grande porte em vários países da América do Sul.'), Document(id='c03bdef7-8eeb-4483-a863-520626bbc9a0', metadata={}, page_content='A pessoa em questão chama-se Luana Maria Silva e é uma arquiteta experiente. Ela estudou administração de empresas na PUC-SP e posteriormente seguiu sua paixão pela arquitetura, especializando-se em projectos de sustainability. Com uma carreira de 10 anos de experiência, Luana já participou de diversos projetos sustentáveis em todo o Brasil e desenvol

In [ ]:
for i in result:
  print(i)
  print()

page_content='O nome completo da pessoa é João Pedro Silva, um arquiteto com formação em Engenharia de Construção Civil pelo Instituto Tecnológico de Aeronáutica, onde obteve sua graduação em 2010. Ele trabalhou em várias empresas de construção civil, incluindo o cargo de gerente de projetos em uma empresa líder no setor por 5 anos antes de criar sua própria empresa de arquitetura em 2015. Hoje, João Pedro é um renomado arquiteto com projetos de grande porte em vários países da América do Sul.'

page_content='A pessoa em questão chama-se Luana Maria Silva e é uma arquiteta experiente. Ela estudou administração de empresas na PUC-SP e posteriormente seguiu sua paixão pela arquitetura, especializando-se em projectos de sustainability. Com uma carreira de 10 anos de experiência, Luana já participou de diversos projetos sustentáveis em todo o Brasil e desenvolveu uma paixão pela criação de espaços que promovem o bem-estar dos seres humanos e do meio ambiente.'

page_content='Alexandra Silv

In [ ]:
result = vectorstore.similarity_search_with_score("Preciso de pessoas que trabalham na area de humanas", k=10)

In [ ]:
for i in result:
  print(i)
  print()

(Document(id='daa99263-e4ec-401e-a378-84a9ca898a08', metadata={}, page_content='A pessoa em questão é Luciana Cristina Silva, que trabalha como designer gráfico. Formada em Belas Artes pela Universidade Estadual de Campinas, Luciana também realizou um mestrado em Marketing pela Pontifícia Universidade Católica de São Paulo. Com cerca de 8 anos de experiência na área, ela desenvolveu habilidades em projetos de arte, marketing e comunicação. Atualmente, Luciana trabalha para uma agência de publicidade de São Paulo, colaborando com projetos inovadores e criativos.'), np.float32(1.109623))

(Document(id='10c1e0ae-fa30-4f33-9682-7e73b9dd1e8c', metadata={}, page_content='Encontrei um exemplo de pessoa aleatória: Maria Eduarda Silva é uma psicóloga com 32 anos, formada em Psicologia pela Universidade Federal do Rio de Janeiro (UFRJ) e com especialização em Terapia de Família. Ela trabalhou como assistente social em uma clínica de recuperação de toxicômanos por cerca de 5 anos e atualmente atu

In [ ]:
# Exemplo com Agents

In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent


@tool
def vector_search(query):
    """ Use essa funcao para buscar os dados no banco vetorial perguntando o que voce precisa saber """
    k = 10
    results = vectorstore.similarity_search(query, k=k)

    textos = [doc.page_content for doc in results]

    # junta tudo em uma string só
    resposta = "\n\n".join(textos)

    return resposta


agent = create_agent(llm, tools=[vector_search])

In [ ]:
# The system prompt will be set dynamically based on context
result = agent.invoke(
    {"messages": [

        {"role": "system", "content": """
          voce e um headhunter, preciso de ajuda pra procurar pessoas.
          use a tool de vector_search quantas vezes necessario pra buscar a informacao

        """},

         {"role": "user", "content": "preciso de 10 engenheiros pra minha obra"}

        ]})

In [ ]:
result['messages']

5

In [ ]:
print(result['messages'][-1].content)

Aqui estão 10 engenheiros civis que podem ser adequados para sua obra:

1. **Eduardo Silva**
   - Idade: 35 anos
   - Formação: Engenharia Civil - Universidade Federal do Rio de Janeiro
   - Experiência: Mais de 8 anos em projetos de infraestrutura, incluindo usinas hidrelétricas e reformas de aeroportos.

2. **Eduardo Henrique Silva**
   - Idade: 32 anos
   - Formação: Engenharia Civil - UFRJ, especialização em gestão de infraestrutura
   - Experiência: Gerente de projetos em construção civil, atuou em pontes, rodovias e edifícios comerciais.

3. **Eduardo Silva**
   - Idade: 35 anos
   - Formação: Engenharia de Construções - UFRJ
   - Experiência: 15 anos na área, 5 anos como gerente de projetos, atuou em pontes e revitalização de edifícios históricos.

4. **Luana Beatriz dos Santos**
   - Idade: 32 anos
   - Formação: Engenharia Civil - UFRJ
   - Experiência: Mais de 7 anos como gerente de projetos em obras públicas, supervisão de estradas e pontes.

5. **João Pedro Silva**
   - Ida